# Class 04 · Part 2: Comparing datasets

In part 1, `class04_get_frogs_data.ipynb`, you downloaded the precipitation of 2020 from two products,
GPCP and IMERG. Now you

1. draw **maps** of both, with the **same colours**,
2. map their **difference**,
3. compare them with **numbers**, and
4. plot their **time series** as two curves.

The difference between products is a measure of how **uncertain** our knowledge of precipitation is.

## 1. Packages and data

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from utils import download_frogs

gpcp = xr.open_dataset(download_frogs("GPCP_CDR_v1.3_yes_enforced", 2020))["rain"]
imerg = xr.open_dataset(download_frogs("IMERG_V07B_FC", 2020))["rain"]

## 2. Maps with the same colours

The mean of all days of 2020 gives a map of each product. To compare two maps, they need the **same colour
scale**: `vmin` and `vmax` fix it. Otherwise the same colour would mean different values.

`Robinson` is a projection for the whole world.

In [ ]:
gpcp_year = gpcp.mean("time")
imerg_year = imerg.mean("time")

fig, axes = plt.subplots(
    1, 2, figsize=(16, 4), subplot_kw={"projection": ccrs.Robinson()}
)
for ax, data, name in [(axes[0], gpcp_year, "GPCP"), (axes[1], imerg_year, "IMERG")]:
    data.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap="Blues",
        vmin=0,
        vmax=10,
        cbar_kwargs={"label": "precipitation / mm per day", "shrink": 0.7},
    )
    ax.coastlines(linewidth=0.5)
    ax.set_title(f"{name}, mean of 2020")

fig.savefig("/app/pages/04_comparing_datasets/maps.png", dpi=100, bbox_inches="tight")
plt.show()

## 3. The difference

Because the grids are the same, you can simply subtract: `imerg_year - gpcp_year`. For a difference, use
colours that are white at 0 and go in two directions: `vmin=-3, vmax=3` keeps 0 in the middle.

Green: IMERG has more rain than GPCP; brown: less.

In [ ]:
difference = imerg_year - gpcp_year

fig = plt.figure(figsize=(10, 5))
ax = plt.axes(projection=ccrs.Robinson())
difference.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="BrBG",
    vmin=-3,
    vmax=3,
    cbar_kwargs={"label": "IMERG − GPCP / mm per day", "shrink": 0.7},
)
ax.coastlines(linewidth=0.5)
ax.set_title("Difference IMERG − GPCP, mean of 2020")

fig.savefig(
    "/app/pages/04_comparing_datasets/difference.png", dpi=100, bbox_inches="tight"
)
plt.show()

Where do they agree, and where not? Look at the tropics (near the equator), where it rains most, and at
the regions near the poles.

## 4. Numbers

The **global mean**, weighted by the area of the cells as in class 03:

In [ ]:
weights = np.cos(np.deg2rad(gpcp.lat))

gpcp_global = float(gpcp_year.weighted(weights).mean())
imerg_global = float(imerg_year.weighted(weights).mean())

print(f"GPCP:  {gpcp_global:.2f} mm per day")
print(f"IMERG: {imerg_global:.2f} mm per day")
print(f"difference: {100 * (imerg_global - gpcp_global) / gpcp_global:.1f} %")

The global means are close. But the map showed large differences in some regions: positive and negative
differences cancel in the global mean. The mean difference in **bands of latitude** shows it:

In [ ]:
bands = [(-90, -60), (-60, -30), (-30, -10), (-10, 10), (10, 30), (30, 60), (60, 90)]

for south, north in bands:
    band = difference.sel(lat=slice(south, north))
    print(
        f"{south:4d}° to {north:3d}°: {float(band.weighted(weights).mean()):+.2f} mm per day"
    )

## 5. Time series: two curves

The precipitation of **Europe** (the same box as in class 03), month by month, for both products:

In [ ]:
def europe_monthly(rain):
    europe = rain.sel(lat=slice(35, 72), lon=slice(-25, 45))
    daily = europe.weighted(weights).mean(("lat", "lon"))
    return daily.resample(time="MS").mean()


gpcp_europe = europe_monthly(gpcp)
imerg_europe = europe_monthly(imerg)

fig, ax = plt.subplots(figsize=(10, 4))
gpcp_europe.plot(ax=ax, marker="o", label="GPCP")
imerg_europe.plot(ax=ax, marker="o", label="IMERG")
ax.set_ylabel("precipitation / mm per day")
ax.set_xlabel("month of 2020")
ax.set_title("Monthly mean precipitation of Europe (35–72°N, 25°W–45°E)")
ax.legend()

fig.savefig(
    "/app/pages/04_comparing_datasets/timeseries.png", dpi=100, bbox_inches="tight"
)
plt.show()

**Try it:** which month has the largest difference? `(imerg_europe - gpcp_europe)` gives the differences;
`abs(...).idxmax()` the month of the largest one.

In [ ]:
differences = imerg_europe - gpcp_europe
print(differences.round(2).to_series())
print("largest difference in:", str(abs(differences).idxmax().values)[:7])

## 6. Exercise: a third product

Add the product you downloaded in part 1 (e.g. `ERA5`):

1. Open it, and draw the difference of the third product minus GPCP, as in section 3.
2. With three products, the **spread** tells how much they disagree: put them together with
   `xr.concat([gpcp_year, imerg_year, third_year], dim="product")` and take `.std("product")`, the standard
   deviation across the products. Map it, and save it as `/app/pages/04_comparing_datasets/spread.png`.
3. The time series of Europe with **three curves** (use `europe_monthly`), saved as
   `/app/pages/04_comparing_datasets/timeseries_three.png`.
4. Where is the spread largest? Is the reanalysis closer to GPCP or to IMERG?

In [ ]:
# 1. the third product and its difference to GPCP

In [ ]:
# 2. the spread of the three products

In [ ]:
# 3. Europe with three curves

## 7. Your page

Open `pages/04_comparing_datasets/my_page.rst` in JupyterLab, write your name, and describe what you found:
where the products agree and where they do not. Look at it on http://localhost:8000 → *Classes → 04*.